## 1. Load your recording















In [ ]:
# Prepare packages inside browser Python (nothing is installed on your computer).
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install([
        'numpy', 'scipy', 'matplotlib',
        'ipywidgets>=8.1.3,<9', 'plotly>=6,<7', 'nbformat>=5,<6'
    ])

import io
import json
import zipfile
import numpy as np
import ipywidgets as widgets
from IPython.display import display

heading = widgets.HTML("""
<div style="margin-bottom:12px">
    <h2 style="margin:0 0 8px">Start here · Load your recording</h2>
    <p style="margin:0; font-size:15px">
        Select the <b>ZIP recording exported from the virtual rig</b>.
        Your voltage data and stimulus information will be loaded automatically.
        No need to unzip the file yourself.
    </p>
</div>
""")

uploader = widgets.FileUpload(
    accept='.zip',
    multiple=False,
    description='Upload ZIP data',
    icon='upload',
    button_style='primary',
    layout=widgets.Layout(width='260px', height='48px')
)

status = widgets.HTML()
details = widgets.Output()

metadata = stimuli = voltage = fs = filename = None

def load_recording(change):
    global metadata, stimuli, voltage, fs, filename

    if not uploader.value:
        return

    metadata = stimuli = voltage = fs = filename = None
    details.clear_output()
    status.value = '<p style="color:#555">Reading and unpacking recording…</p>'

    try:
        # Compatible with ipywidgets 7 and 8
        if isinstance(uploader.value, dict):
            selected_name, item = next(iter(uploader.value.items()))
        else:
            item = uploader.value[0]
            selected_name = item['name']

        with zipfile.ZipFile(io.BytesIO(bytes(item['content']))) as z:
            new_metadata = json.loads(z.read('metadata.json'))
            new_stimuli = json.loads(z.read('stimuli.json'))
            new_voltage = np.frombuffer(
                z.read('voltage.bin'), dtype='<f4'
            ).copy()

        new_fs = float(new_metadata['sampling_rate_hz'])
        if not np.isfinite(new_fs) or new_fs <= 0:
            raise ValueError('The recording has an invalid sampling rate.')

        metadata, stimuli = new_metadata, new_stimuli
        voltage, fs, filename = new_voltage, new_fs, selected_name

        status.value = """
        <p style="color:#237744; font-size:15px">
            <b>✓ Recording loaded. Continue to the next cell.</b>
        </p>
        """

        with details:
            print(f'File:          {filename}')
            print(f'Duration:      {len(voltage) / fs:.1f} seconds')
            print(f'Sampling rate: {fs:,.0f} Hz')
            print(f'Samples:       {len(voltage):,}')

    except Exception as error:
        status.value = """
        <p style="color:#b42318">
            <b>Could not load the recording.</b>
            Select a ZIP exported from the virtual rig.
        </p>
        """
        with details:
            print(str(error))

uploader.observe(load_recording, names='value')

display(widgets.VBox(
    [heading, uploader, status, details],
    layout=widgets.Layout(
        padding='20px',
        border='1px solid #dce3eb',
        max_width='750px'
    )
))

## 2. Inspect the raw recording

The black trace shows the unfiltered extracellular voltage. Coloured backgrounds
mark grating presentations; colours indicate stimulus orientation.

**Explore:** Drag to zoom into a short interval. Use the pan tool to move through
the recording. Double-click to reset the axes.

**Question:** Can you identify individual spikes? What makes them difficult to see?

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

t = np.arange(len(voltage)) / fs

# Identify grating presentations and assign orientation colours
gratings = [
    s for s in stimuli
    if s['parameters']['type'] == 'grating'
]

orientations = sorted({
    s['parameters']['orientation'] for s in gratings
})

palette = sample_colorscale(
    'HSV',
    np.linspace(0, 1, len(orientations), endpoint=False)
) if orientations else []

colors = dict(zip(orientations, palette))

# Raw voltage
fig_raw = go.Figure()

fig_raw.add_trace(go.Scattergl(
    x=t,
    y=voltage,
    mode='lines',
    line=dict(color='black', width=0.7),
    name='Raw voltage',
    showlegend=False,
    hovertemplate='Time: %{x:.4f} s<br>Voltage: %{y:.1f} µV<extra></extra>'
))

# Stimulus backgrounds
stimulus_shapes = [
    dict(
        type='rect',
        xref='x', yref='paper',
        x0=s['onset_s'], x1=s['offset_s'],
        y0=0, y1=1,
        fillcolor=colors[s['parameters']['orientation']],
        opacity=0.18,
        line=dict(width=0),
        layer='below'
    )
    for s in gratings
]

fig_raw.update_layout(shapes=stimulus_shapes)

# Orientation legend
for angle in orientations:
    fig_raw.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=10, color=colors[angle]),
        name=f'{angle:g}°'
    ))

fig_raw.update_layout(
    title='2. Raw extracellular recording',
    xaxis_title='Time (s)',
    yaxis_title='Voltage (µV)',
    height=500,
    template='plotly_white',
    dragmode='zoom',
    hovermode='closest',
    legend_title='Grating orientation',
    margin=dict(l=65, r=20, t=60, b=55)
)

fig_raw.show(config={'displaylogo': False, 'scrollZoom': True})

## 3. Filter the recording

A 300 Hz high-pass filter attenuates slow voltage fluctuations, helping us
inspect the faster signals associated with spikes. The original data remain
available in `voltage`; the filtered data are stored in `voltage_filtered`.

The filter is applied forward and backward to avoid introducing a phase delay.

**Explore:** Zoom into the same interval as before.

**Question:** What changed? Does filtering alone tell us which neuron produced
each spike?

In [ ]:
from scipy.signal import butter, sosfiltfilt

cutoff_hz = 300

sos = butter(
    3,
    cutoff_hz,
    btype='highpass',
    fs=fs,
    output='sos'
)

# Filter the complete recording before zooming into individual intervals
voltage_filtered = sosfiltfilt(sos, voltage)

# Reuse the same stimulus shading, colours and layout
fig_filtered = go.Figure(fig_raw)

fig_filtered.data[0].y = voltage_filtered
fig_filtered.data[0].name = 'Filtered voltage'

fig_filtered.update_layout(
    title=f'3. High-pass filtered recording — {cutoff_hz} Hz'
)

fig_filtered.show(config={'displaylogo': False, 'scrollZoom': True})

## 4. Detect spikes using a threshold

The red dashed line is an automatically calculated negative detection threshold:
4.5 times a robust estimate of the noise level below the signal median.

The lower panel marks detected events as short black bars. Both panels share
the same time axis, so zooming shows the voltage and detected events together.

**Question:** Which voltage deflections are detected? Could noise cross the
threshold, or small spikes remain undetected?

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import find_peaks

# Automatically calculate the negative detection threshold
threshold_factor = 4.5
signal_median = np.median(voltage_filtered)
noise_sigma = np.median(
    np.abs(voltage_filtered - signal_median)
) / 0.67449

threshold = signal_median - threshold_factor * noise_sigma

# Detect negative peaks with a minimum separation of 1 ms
min_separation_ms = 1.0

spike_indices, _ = find_peaks(
    -voltage_filtered,
    height=-threshold,
    distance=max(1, round(min_separation_ms / 1000 * fs))
)

spike_times = spike_indices / fs
t = np.arange(len(voltage_filtered)) / fs

print(f'Estimated noise: {noise_sigma:.2f} µV')
print(f'Threshold:       {threshold:.2f} µV')
print(f'Detected events: {len(spike_times):,}')

# Filtered voltage above, continuous spike raster below
fig_detection = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.78, 0.22],
    vertical_spacing=0.06
)

fig_detection.add_trace(go.Scattergl(
    x=t,
    y=voltage_filtered,
    mode='lines',
    line=dict(color='black', width=0.7),
    showlegend=False,
    hovertemplate='Time: %{x:.4f} s<br>Voltage: %{y:.1f} µV<extra></extra>'
), row=1, col=1)

# Thin vertical ticks mark detected spikes
fig_detection.add_trace(go.Scattergl(
    x=spike_times,
    y=np.zeros(len(spike_times)),
    mode='markers',
    marker=dict(
        symbol='line-ns',
        size=12,
        color='black',
        line=dict(color='black', width=0.5)
    ),
    showlegend=False,
    hovertemplate='Detected event: %{x:.4f} s<extra></extra>'
), row=2, col=1)

# Orientation shading in both panels
shapes = []

for row in [1, 2]:
    axis_suffix = '' if row == 1 else '2'

    for s in gratings:
        shapes.append(dict(
            type='rect',
            xref=f'x{axis_suffix}',
            yref=f'y{axis_suffix} domain',
            x0=s['onset_s'],
            x1=s['offset_s'],
            y0=0,
            y1=1,
            fillcolor=colors[s['parameters']['orientation']],
            opacity=0.18,
            line=dict(width=0),
            layer='below'
        ))

fig_detection.update_layout(shapes=shapes)

fig_detection.add_hline(
    y=threshold,
    row=1,
    col=1,
    line_color='#d62728',
    line_dash='dash',
    line_width=1.5,
    annotation_text=f'Threshold: {threshold:.1f} µV',
    annotation_position='top right'
)

# Orientation legend
for angle in orientations:
    fig_detection.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(size=10, color=colors[angle]),
        name=f'{angle:g}°'
    ), row=1, col=1)

fig_detection.update_layout(
    title='4. Threshold detection and continuous spike raster',
    height=600,
    template='plotly_white',
    dragmode='zoom',
    hovermode='closest',
    legend_title='Grating orientation',
    margin=dict(l=75, r=20, t=65, b=55)
)

fig_detection.update_yaxes(
    title_text='Voltage (µV)',
    row=1,
    col=1
)

fig_detection.update_yaxes(
    title_text='Events',
    range=[-1, 1],
    showticklabels=False,
    showgrid=False,
    zeroline=False,
    fixedrange=True,
    row=2,
    col=1
)

fig_detection.update_xaxes(
    title_text='Time (s)',
    row=2,
    col=1
)

fig_detection.show(config={
    'displaylogo': False,
    'scrollZoom': True
})

## 5. Compare responses across trials

Each row shows one grating presentation, aligned to stimulus onset at **0 s**.
Trials are grouped by grating orientation. Each black tick marks a detected event.

The bar and dotted lines indicate the stimulus period. The intervals
before and after show baseline and post-stimulus activity.

**Question:** Which orientation produces the strongest response? How consistent
is the response across repeated presentations?

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Use only gratings presented by the experimental script.
gratings = [
    s for s in stimuli
    if s.get('source') == 'protocol'
    and s['parameters']['type'] == 'grating'
]

if not gratings:
    raise ValueError('No scripted grating presentations were found.')

orientations = sorted({
    s['parameters']['orientation'] for s in gratings
})

pre_stimulus_s = 0.5
post_stimulus_s = 0.5

trial_groups = {
    angle: [
        (trial_number, s)
        for trial_number, s in enumerate(gratings, start=1)
        if s['parameters']['orientation'] == angle
    ]
    for angle in orientations
}

durations = np.array([
    s['offset_s'] - s['onset_s'] for s in gratings
])

x_min = -pre_stimulus_s
x_max = durations.max() + post_stimulus_s
recording_end = len(voltage_filtered) / fs

same_duration = np.allclose(
    durations, durations[0], rtol=0, atol=1 / fs
)

fig_raster = go.Figure()

backgrounds = []
timing_lines = []
labels = []
tick_positions = []
tick_labels = []

boundary_style = dict(
    color='rgba(40,40,40,0.4)',
    width=1,
    dash='dot'
)

row = 0

for angle in orientations:
    trials = trial_groups[angle]
    first_row = row

    spike_x = []
    spike_y = []
    trial_hover = []

    for trial_number, s in trials:
        onset = s['onset_s']
        duration = s['offset_s'] - onset

        start = np.searchsorted(
            spike_times,
            max(0, onset - pre_stimulus_s)
        )
        stop = np.searchsorted(
            spike_times,
            min(recording_end, s['offset_s'] + post_stimulus_s),
            side='right'
        )

        relative_times = spike_times[start:stop] - onset

        spike_x.extend(relative_times)
        spike_y.extend([row] * len(relative_times))
        trial_hover.extend([trial_number] * len(relative_times))

        # If durations differ, mark the offset separately for each trial
        if not same_duration:
            timing_lines.append(dict(
                type='line',
                xref='x', yref='y',
                x0=duration, x1=duration,
                y0=row - 0.45, y1=row + 0.45,
                line=boundary_style,
                layer='above'
            ))

        tick_positions.append(row)
        tick_labels.append(str(trial_number))
        row += 1

    last_row = row - 1

    # Uniform background throughout each orientation group
    backgrounds.append(dict(
        type='rect',
        xref='x', yref='y',
        x0=x_min, x1=x_max,
        y0=first_row - 0.5, y1=last_row + 0.5,
        fillcolor=colors[angle],
        opacity=0.12,
        line=dict(width=0),
        layer='below'
    ))

    labels.append(dict(
        x=1.015, xref='paper',
        y=(first_row + last_row) / 2, yref='y',
        text=f'<b>{angle:g}°</b>',
        showarrow=False,
        xanchor='left',
        font=dict(size=13, color='#333333')
    ))

    fig_raster.add_trace(go.Scattergl(
        x=spike_x,
        y=spike_y,
        customdata=trial_hover,
        mode='markers',
        marker=dict(
            symbol='line-ns',
            size=9,
            color='black',
            line=dict(color='black', width=0.6)
        ),
        showlegend=False,
        hovertemplate=(
            f'Orientation: {angle:g}°'
            '<br>Trial: %{customdata}'
            '<br>Time from onset: %{x:.3f} s'
            '<extra></extra>'
        )
    ))

    row += 1

# Common stimulus onset: faint dotted line across all trials
timing_lines.append(dict(
    type='line',
    xref='x', yref='paper',
    x0=0, x1=0,
    y0=0, y1=1,
    line=boundary_style,
    layer='above'
))

# Common offset, when all stimulus durations are equal
if same_duration:
    timing_lines.append(dict(
        type='line',
        xref='x', yref='paper',
        x0=durations[0], x1=durations[0],
        y0=0, y1=1,
        line=boundary_style,
        layer='above'
    ))

# Stimulus bar above the raster
bar_end = durations[0] if same_duration else durations.min()

timing_lines.append(dict(
    type='line',
    xref='x', yref='paper',
    x0=0, x1=bar_end,
    y0=1.035, y1=1.035,
    line=dict(color='#333333', width=6)
))

if not same_duration:
    timing_lines.append(dict(
        type='line',
        xref='x', yref='paper',
        x0=durations.min(), x1=durations.max(),
        y0=1.035, y1=1.035,
        line=dict(color='#777777', width=3, dash='dash')
    ))

labels.append(dict(
    x=bar_end / 2, xref='x',
    y=1.065, yref='paper',
    text=(
        'Grating stimulus' if same_duration
        else 'Grating stimulus · offset varies across trials'
    ),
    showarrow=False,
    yanchor='bottom',
    font=dict(size=13, color='#333333')
))

fig_raster.update_layout(
    title=dict(
        text='5. Trial-aligned spike raster — grouped by orientation',
        y=0.98,
        yanchor='top'
    ),
    shapes=backgrounds + timing_lines,
    annotations=labels,
    height=max(480, 200 + 17 * row),
    template='plotly_white',
    dragmode='zoom',
    hovermode='closest',
    margin=dict(l=75, r=85, t=125, b=65),
    plot_bgcolor='white'
)

fig_raster.update_xaxes(
    title_text='Time from stimulus onset (s)',
    range=[x_min, x_max],
    showgrid=False,
    zeroline=False
)

fig_raster.update_yaxes(
    title_text='Trial (original presentation number)',
    tickmode='array',
    tickvals=tick_positions,
    ticktext=tick_labels,
    range=[row - 1.3, -0.8],
    showgrid=False,
    zeroline=False,
    fixedrange=True
)

fig_raster.show(config={
    'displaylogo': False,
    'scrollZoom': True
})

## 6. Orientation tuning curve

For each trial, count the detected events during the stimulus and divide by
the stimulus duration to calculate the firing rate.

Points show the mean across trials; error bars show the standard error of
the mean (SEM).

**Question:** Which orientation produces the highest mean firing rate?

In [ ]:
import numpy as np
import plotly.graph_objects as go

mean_rates = []
sem_rates = []
trial_counts = []

for angle in orientations:
    trials = [
        s for s in gratings
        if s['parameters']['orientation'] == angle
    ]

    rates = []

    for s in trials:
        onset = s['onset_s']
        offset = s['offset_s']

        # Use only complete stimulus periods
        if onset < 0 or offset > len(voltage_filtered) / fs:
            continue

        duration = offset - onset
        if duration <= 0:
            continue

        # Count events in [onset, offset)
        start = np.searchsorted(spike_times, onset, side='left')
        stop = np.searchsorted(spike_times, offset, side='left')

        rates.append((stop - start) / duration)

    rates = np.asarray(rates)

    trial_counts.append(len(rates))
    mean_rates.append(np.mean(rates) if len(rates) else np.nan)
    sem_rates.append(
        np.std(rates, ddof=1) / np.sqrt(len(rates))
        if len(rates) > 1 else np.nan
    )

fig_tuning = go.Figure(go.Scatter(
    x=orientations,
    y=mean_rates,
    mode='lines+markers',
    line=dict(color='#333333', width=1.5),
    marker=dict(
        size=10,
        color=[colors[angle] for angle in orientations],
        line=dict(color='#333333', width=0.7)
    ),
    error_y=dict(
        type='data',
        array=sem_rates,
        visible=True,
        color='#333333',
        thickness=1.2,
        width=5
    ),
    customdata=np.column_stack([sem_rates, trial_counts]),
    hovertemplate=(
        'Orientation: %{x}°'
        '<br>Mean rate: %{y:.2f} events/s'
        '<br>SEM: %{customdata[0]:.2f} events/s'
        '<br>Trials: %{customdata[1]:.0f}'
        '<extra></extra>'
    )
))

fig_tuning.update_layout(
    title='6. Orientation tuning — mean ± SEM',
    xaxis_title='Grating orientation (°)',
    yaxis_title='Detected event rate (events/s)',
    height=450,
    template='plotly_white',
    showlegend=False,
    margin=dict(l=75, r=25, t=65, b=65)
)

fig_tuning.update_xaxes(
    tickmode='array',
    tickvals=orientations,
    ticktext=[f'{angle:g}°' for angle in orientations],
    showgrid=False
)

fig_tuning.update_yaxes(rangemode='tozero')

fig_tuning.show(config={'displaylogo': False})